# HR Analytics Dashboard
Upload `HR_Data.xlsx` in Step 1, then run each cell.

**KPIs:** Employee Count | Attrition Count | Attrition Rate | Active Employees | Average Age

**Charts:** Gender | Department | Age Group | Job Satisfaction | Education | Gender + Age

## Step 1 — Load Data

In [ ]:
!pip install openpyxl --quiet
import pandas as pd, matplotlib.pyplot as plt
from google.colab import files

f = files.upload()
df = pd.read_excel(list(f.keys())[0])
df['Left'] = (df['Attrition']=='Yes').astype(int)
print('Employees:', len(df))

---
## KPI Cards

In [ ]:
total, left = len(df), df['Left'].sum()

kpis = [('Employee Count', total, '#4472C4'),
        ('Attrition Count', left, '#E05C5C'),
        ('Attrition Rate', f'{left/total*100:.1f}%', '#E05C5C'),
        ('Active Employees', total-left, '#70AD47'),
        ('Average Age', f'{df.Age.mean():.1f}', '#FFC000')]

fig, axes = plt.subplots(1, 5, figsize=(16, 2.5))
for ax, (lbl, val, col) in zip(axes, kpis):
    ax.set_facecolor(col)
    ax.text(0.5, 0.6, str(val), ha='center', fontsize=18, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.2, lbl,      ha='center', fontsize=8,  color='white',     transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## Chart 1 — Attrition by Gender

In [ ]:
g = df[df.Attrition=='Yes'].Gender.value_counts()

fig, (a1,a2) = plt.subplots(1, 2, figsize=(8, 3))
a1.bar(g.index, g.values, color=['#4472C4','#E05C5C'], width=0.4)
a1.set_title('Count'); a1.set_ylabel('Employees')
a2.pie(g, labels=g.index, autopct='%1.1f%%', colors=['#4472C4','#E05C5C'],
       wedgeprops=dict(width=0.5))
a2.set_title('Share')
fig.suptitle('Attrition by Gender', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Chart 2 — Department-wise Attrition

In [ ]:
d = df.groupby('Department')['Left'].agg(['sum','count'])
d['Rate'] = (d['sum']/d['count']*100).round(1)
c = ['#4472C4','#E05C5C','#70AD47']

fig, (a1,a2) = plt.subplots(1, 2, figsize=(10, 3))
a1.bar(d.index, d['sum'],  color=c); a1.set_title('Count');  a1.set_ylabel('Employees')
a2.bar(d.index, d['Rate'], color=c); a2.set_title('Rate %'); a2.set_ylabel('%')
fig.suptitle('Department-wise Attrition', fontweight='bold')
plt.tight_layout(); plt.show()
print(d.rename(columns={'sum':'Left','count':'Total'}))

---
## Chart 3 — Employees by Age Group

In [ ]:
df['AgeGrp'] = pd.cut(df.Age, bins=[18,25,35,45,55,60],
                      labels=['18-25','26-35','36-45','46-55','55+'])
tot = df.groupby('AgeGrp', observed=True).size()
att = df[df.Attrition=='Yes'].groupby('AgeGrp', observed=True).size()
c   = ['#4472C4','#E05C5C','#70AD47','#FFC000','#9B59B6']

fig, (a1,a2) = plt.subplots(1, 2, figsize=(11, 3))
a1.bar(tot.index, tot.values, color=c); a1.set_title('Total Employees')
a2.bar(att.index, att.values, color='#E05C5C'); a2.set_title('Attrition Count')
fig.suptitle('Employees by Age Group', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Chart 4 — Job Satisfaction Ratings

In [ ]:
df['Sat'] = df['Job Satisfaction'].map({1:'Low',2:'Medium',3:'High',4:'Very High'})
sat = df.groupby(['Sat','Attrition']).size().unstack().fillna(0)
sat = sat.reindex(['Low','Medium','High','Very High'])

sat.plot(kind='bar', color=['#4472C4','#E05C5C'], edgecolor='white',
         figsize=(7, 3), rot=0)
plt.title('Job Satisfaction vs Attrition', fontweight='bold')
plt.ylabel('Employees'); plt.legend(title='Attrition')
plt.tight_layout(); plt.show()

---
## Chart 5 — Education Field-wise Attrition

In [ ]:
edu = df.groupby('Education Field')['Left'].agg(['sum','count'])
edu['Rate'] = (edu['sum']/edu['count']*100).round(1)
edu = edu.sort_values('sum')

fig, (a1,a2) = plt.subplots(1, 2, figsize=(12, 3))
a1.barh(edu.index, edu['sum'],  color='#4472C4'); a1.set_title('Count')
a2.barh(edu.index, edu['Rate'], color='#E05C5C'); a2.set_title('Rate %')
fig.suptitle('Education Field-wise Attrition', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Chart 6 — Attrition Rate by Gender & Age Group

In [ ]:
ag = df.groupby(['AgeGrp','Gender'], observed=True)['Left'].agg(['sum','count'])
ag['Rate'] = (ag['sum']/ag['count']*100).round(1)
pivot = ag['Rate'].unstack().fillna(0)

pivot.plot(kind='bar', color=['#E05C5C','#4472C4'], edgecolor='white',
           figsize=(8, 3), rot=0)
plt.title('Attrition Rate by Gender & Age Group', fontweight='bold')
plt.ylabel('Rate (%)'); plt.legend(title='Gender')
plt.tight_layout(); plt.show()

---
## Summary

In [ ]:
print('Employee Count   :', len(df))
print('Attrition Count  :', df.Left.sum())
print('Attrition Rate   :', f"{df.Left.mean()*100:.1f}%")
print('Active Employees :', len(df) - df.Left.sum())
print('Average Age      :', round(df.Age.mean(), 1))
print()
print('Top Dept  :', df.groupby('Department')['Left'].mean().idxmax())
print('Top Role  :', df.groupby('Job Role')['Left'].mean().idxmax())
print('Top Edu   :', df.groupby('Education Field')['Left'].mean().idxmax())